# F2-vectors — Practice p06 — Solution

In [ ]:
import numpy as np

SEED = 20260804
rng = np.random.default_rng(SEED)

In [ ]:
X = rng.normal(size=(200, 3))
u = np.array([2.0, -1.0, 2.0]) / 3.0    # a unit vector: sqrt(4 + 1 + 4) / 3 = 1

print(X.shape, u, np.linalg.norm(u))

## Part (a)

In [ ]:
def project_rows(X, u):
    dots = np.sum(X * u, axis=1)     # (200,): the dot product of every row with u
    return dots[:, np.newaxis] * u   # (200, 1) * (3,) broadcasts to (200, 3)


P = project_rows(X, u)
print(P.shape)
print(P[0])

`X * u` broadcasts `u` across all 200 rows, and the `axis=1` sum turns each
row into its dot product with `u` — 200 dot products in one line.
Because `u` is a unit vector, each projection is just that dot product times
`u`; reshaping `dots` to a `(200, 1)` column lets broadcasting scale `u` by a
different amount for every row at once.

## Part (b)

In [ ]:
def cosine_rows(X, u):
    dots = np.sum(X * u, axis=1)
    row_norms = np.sqrt(np.sum(X**2, axis=1))
    return dots / (row_norms * np.linalg.norm(u))


cos_all = cosine_rows(X, u)
check_range = bool(np.all((cos_all >= -1.0) & (cos_all <= 1.0)))
print(cos_all[:3], check_range)

The same axis-sum trick computes all 200 numerators, and a second axis sum
(of squares) gives all 200 row norms.
Dividing elementwise finishes 200 cosine similarities without a single loop,
and every value lands in $[-1, 1]$ as the lesson's angle argument
guarantees.

## Part (c)

In [ ]:
R = X - P
check_orth = bool(np.allclose(np.sum(R * u, axis=1), 0.0, atol=1e-9, rtol=0))
print(check_orth)

`np.sum(R * u, axis=1)` is the dot product of every residual row with `u`,
all at once; `np.allclose` confirms the whole batch is zero to floating-point
precision.
This is the residual-orthogonality property from the lesson, holding for all
200 vectors simultaneously.

### Answer check

In [ ]:
assert P.shape == (200, 3)
assert cos_all.shape == (200,)
assert check_range is True
assert check_orth is True
# Row 0 against the single-vector formulas from the lesson:
a0 = X[0]
assert np.allclose(P[0], (a0 @ u) * u, atol=1e-9, rtol=0)
assert np.allclose(P[0], np.array([-0.48564109, 0.24282055, -0.48564109]), atol=1e-6, rtol=0)
assert np.isclose(cos_all[0], (a0 @ u) / (np.linalg.norm(a0) * np.linalg.norm(u)), atol=1e-9, rtol=0)
assert np.isclose(cos_all[0], -0.54356423, atol=1e-6, rtol=0)
assert np.isclose(np.sum(np.sum(X * u, axis=1)), -23.6846007694, atol=1e-6, rtol=0)
print("All checks passed.")